In [ ]:
import ibis
from utils.f_0_dirs import get_data_dirs

table_name = "working_yearly_with_tfp_wave"
dirs = get_data_dirs(segment="descriptives")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)
t_panel = con.table(table_name)

# Plot PACF and ACF
- Relevant time series used for dynamics to determine appropriate lag order

In [ ]:
# Plot the autocorrelation function (ACF) of the forecast errors to visually inspect for any patterns or autocorrelation
from statsmodels.graphics.tsaplots import plot_acf
plt.figure(figsize=(10, 5))
plot_acf(df['diff'], lags=20, ax=plt.gca(), alpha=0.05) 
plt.title('Autocorrelation Function (ACF) of Forecast Errors')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.axhline(0, color='black', linestyle='--')
plt.show()

# Plot the partial autocorrelation function (PACF) of the forecast errors to identify the order of any autoregressive process
from statsmodels.graphics.tsaplots import plot_pacf
plt.figure(figsize=(10, 5))
plot_pacf(df['diff'], lags=20, ax=plt.gca(), alpha=0.05)
plt.title('Partial Autocorrelation Function (PACF) of Forecast Errors')
plt.xlabel('Lag')
plt.ylabel('Partial Autocorrelation')
plt.axhline(0, color='black', linestyle='--')
plt.show()

In [ ]:
%%script true
# Run a BIC on df['unemp_effect'] and choose the best model based on the BIC value. Does it differ from the AR(1) model?

from statsmodels.tsa.ar_model import ar_select_order
import pandas as pd

# from sklearn.linear_model import LinearRegression
# import numpy as np

index_properties = ["registered_number", "year"]
var_properties = ["gva1", "tfp_wav1", "tfp_wav2", "tfp_wav3"]
max_lag = 15
diff_lvl = 1

# --- #
df_ar = (
    t_panel
    .distinct(on=index_properties)
    .select(index_properties + var_properties)
    .drop_null(var_properties)
    .execute()
)
print(f"df of length: {len(df_ar):,}")
df_ar_diffed = df_ar.copy().sort_values("year")
for i in range(diff_lvl):
    for p in var_properties:
        df_ar_diffed[f"D{i + 1}.{p}"] = df_ar_diffed.groupby(level=0)[p].diff()
    print(f"df of length after diff {i + 1}: {len(df_ar_diffed):,}")
df_sample = df_ar_diffed.sample(5)
for row in df_sample.itertuples():
    df_sample_plus = df_ar_diffed.loc[
        (df_ar_diffed["registered_number"] == row.registered_number) &
        ((df_ar_diffed["year"] == row.year - 1) |
        (df_ar_diffed["year"] == row.year + 1))
    ]
df_sample = pd.concat([df_sample, df_sample_plus]).sort_values(["registered_number", "year"])
print(df_sample)

rows_ic = []
for p in var_properties:

    print(f'Analysing {p}:')
    col_name = f"D{diff_lvl}.{p}"
    col = df_ar_diffed[col_name].dropna()
    print(f"Column: {col_name}, length: {len(col):,}")
    if len(col) == 0:
        print(f"Column {col_name} is empty after diffing, skipping...")
        continue

    # # 1. BIC - manual
    # bic_info = []
    # for lag in range(1, max_lag + 1):
    #     df_ar[f'L{lag}.{p}'] = df_ar[p].shift(lag)
    # for lag in range(1, max_lag + 1):
    #     lag_columns = [f'L{l_short}.{p}' for l_short in range(1, lag + 1)]
    #     df_ar_lag = df_ar.copy().dropna()
    #     X = df_ar_lag[lag_columns]
    #     Y = df_ar_lag[p]
        
    #     ols_model = LinearRegression()
    #     ols_model.fit(X, Y)
    #     n = len(df_ar_lag)
    #     k = len(X.columns)
    #     bic = n * np.log(np.sum((Y - ols_model.predict(X)) ** 2) / n) + k * np.log(n)
    #     bic_info.append((lag, bic))

    # bic_df = pd.DataFrame(bic_info, columns=['Lag', 'BIC'])
    # bic_star = bic_df.loc[bic_df['BIC'].idxmin()]
    # print(f'--- BIC selection: AR({bic_star["Lag"]}) with BIC = {bic_star["BIC"]:.2f}')

    # 2. Package BIC
    bic_selection = ar_select_order(col, maxlag=max_lag, ic='bic', glob=False)
    bic_best_lag = bic_selection.ar_lags[-1] if bic_selection.ar_lags else 0
    bic_model = bic_selection.model.fit()
    print(f'--- Package BIC selection: AR({bic_best_lag}) with BIC: {bic_model.bic:.2f}')
    
    # 2. Package AIC
    aic_selection = ar_select_order(col, maxlag=max_lag, ic='aic', glob=False)
    aic_best_lag = aic_selection.ar_lags[-1] if aic_selection.ar_lags else 0
    aic_model = aic_selection.model.fit()
    print(f'--- Package AIC selection: AR({aic_best_lag}) with AIC: {aic_model.aic:.2f}')

    row = {
        'Variable': p,
        'AIC lag': aic_best_lag,
        'AIC loss': aic_model.aic,
        'BIC lag': bic_best_lag,
        'BIC loss': bic_model.bic
    }
    rows_ic.append(row)
df_ic = pd.DataFrame(rows_ic)
df_ic.style.format({
    'AIC loss': '{:,.2f}',
    'BIC loss': '{:,.2f}',
    'AIC lag': '{:.0f}',
    'BIC lag': '{:.0f}'
})
print(df_ic)

df of length: 1,027,368
df of length after diff 1: 1,027,368
        registered_number  year          gva1  tfp_wav1  tfp_wav2  tfp_wav3  \
197780           02656007  2007  69349.960231  3.484250  4.161295  3.289278   
221540           03062983  2008    711.422895  2.986340  2.981069  2.995255   
940468           03313799  2013  11597.276415  2.300526  2.261648  2.965354   
9455             03313799  2014  13887.911688  2.410577  2.379012  2.929549   
1022412          03313799  2015  16040.842282  2.429765  2.389296  3.038730   
124843           03471831  2012   2208.382805  2.964170  2.878259  2.926839   
504170           05120951  2018   2144.946716  3.280700  3.167920  3.305919   

         D1.gva1  D1.tfp_wav1  D1.tfp_wav2  D1.tfp_wav3  
197780       NaN          NaN          NaN          NaN  
221540       NaN          NaN          NaN          NaN  
940468       NaN          NaN          NaN          NaN  
9455         NaN          NaN          NaN          NaN  
1022412      NaN

IndexError: index 0 is out of bounds for axis 0 with size 0

In [27]:
%%script true (failed to run)

import pandas as pd
import statsmodels.api as sm

index_properties = ["registered_number", "year"]
var_properties = ["gva1", "tfp_wav1", "tfp_wav2", "tfp_wav3"]
max_lag = 15
diff_lvl = 1

# --- 1. Isolate Panel Data and set MultiIndex --- #
df_ar = (
    t_panel
    .distinct(on=index_properties)
    .select(index_properties + var_properties)
    .drop_null(var_properties)
    .execute()
    .set_index(index_properties) # Enables safe grouping by firm (level=0)
    .sort_index()
)
print(f"df of length: {len(df_ar):,}")

df_ar_diffed = df_ar.copy()

# --- 2. Safe Differencing & Lagging --- #
for i in range(diff_lvl):
    for p in var_properties:
        col_name = f"D{i}.{p}" if i > 0 else p
        new_col_name = f"D{i + 1}.{p}"
        # Group by level=0 (registered_number) to prevent cross-firm differences
        df_ar_diffed[new_col_name] = df_ar_diffed.groupby(level=0)[col_name].diff()
    print(f"df of length after diff {i + 1}: {len(df_ar_diffed):,}")

for p in var_properties:
    col_name = f"D{diff_lvl}.{p}"
    for lag in range(1, max_lag + 1):
        df_ar_diffed[f'L{lag}.{p}'] = df_ar_diffed.groupby(level=0)[col_name].shift(lag)

# --- 3. Run Built-in AIC/BIC Selection --- #
rows_ic = []
for p in var_properties:
    print(f'\nAnalysing {p}:')
    col_name = f"D{diff_lvl}.{p}"
    
    # Drop rows missing the max lag to ensure fair IC comparisons on the exact same sample size
    lag_columns_full = [f'L{l}.{p}' for l in range(1, max_lag + 1)]
    df_model = df_ar_diffed[[col_name] + lag_columns_full].dropna()
    print(f"Column: {col_name}, length: {len(df_model):,}")

    best_aic, best_bic = float('inf'), float('inf')
    aic_lag, bic_lag = 0, 0

    # Iterate through lags and evaluate via standard OLS
    for lag in range(1, max_lag + 1):
        lag_columns = [f'L{l}.{p}' for l in range(1, lag + 1)]
        
        Y = df_model[col_name]
        X = sm.add_constant(df_model[lag_columns])
        
        # Use built-in statsmodels OLS to extract precise AIC/BIC
        ols_model = sm.OLS(Y, X).fit()
        
        if ols_model.aic < best_aic:
            best_aic = ols_model.aic
            aic_lag = lag
        if ols_model.bic < best_bic:
            best_bic = ols_model.bic
            bic_lag = lag

    print(f'--- Built-in BIC selection: AR({bic_lag}) with BIC: {best_bic:.2f}')
    print(f'--- Built-in AIC selection: AR({aic_lag}) with AIC: {best_aic:.2f}')

    rows_ic.append({
        'Variable': p,
        'AIC lag': aic_lag,
        'AIC loss': best_aic,
        'BIC lag': bic_lag,
        'BIC loss': best_bic
    })

df_ic = pd.DataFrame(rows_ic)
display(df_ic.style.format({
    'AIC loss': '{:,.2f}',
    'BIC loss': '{:,.2f}',
    'AIC lag': '{:.0f}',
    'BIC lag': '{:.0f}'
}))
# Write to CSV
with open(dirs.output_dir / "tse_ic_selection.csv", "w") as f:
    df_ic.to_csv(f, index=False)